Cell line selection based on pancancer dataset and MLMarker scores

In [ ]:
!pip install mlmarker==0.1.6
!pip install --upgrade openpyxl --force-reinstall

In [37]:
import os
import tqdm
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from mlmarker import MLMarker

In [38]:
df = pd.read_excel('TableS2.xlsx', sheet_name='Prot matrix excl single-peptide', skiprows=1, index_col=0,)
df = df.transpose().reset_index()
df.head()

Project_Identifier,index,SIDM00018;K052,SIDM00023;TE-12,SIDM00040;TMK-1,SIDM00041;STS-0421,SIDM00042;PL4,SIDM00043;PCI-4B,SIDM00044;PCI-30,SIDM00045;HSC-39,SIDM00046;H3255,...,SIDM01240;451Lu,SIDM01242;MMAc-SF,SIDM01245;BE-13,SIDM01246;MC-IXC,SIDM01247;Ramos-2G6-4C10,SIDM01248;CGTH-W-1,SIDM01251;H9,SIDM01259;GR-ST,SIDM01261;YMB-1-E,SIDM01265;MM1S
0,P37108;SRP14_HUMAN,7.10955,6.82802,7.01426,5.28591,5.70786,6.73965,6.04591,6.20582,6.53547,...,6.45510,6.72638,6.66480,7.02345,6.99319,6.31631,6.23087,7.00407,6.76532,6.91982
1,Q96JP5;ZFP91_HUMAN,3.41494,4.14346,4.19987,3.35789,NaN,4.50199,3.69356,2.88118,NaN,...,3.81755,NaN,3.60415,3.12879,3.85570,4.86933,2.71686,NaN,3.75777,3.70284
2,Q9Y4H2;IRS2_HUMAN,NaN,2.23781,2.44055,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,3.07386,NaN,NaN,NaN,NaN,NaN,NaN
3,P36578;RL4_HUMAN,7.86661,7.62878,8.12459,7.97268,6.22574,7.47364,7.07092,8.25336,6.47861,...,7.55155,7.80751,6.99447,7.78652,7.32346,7.62433,7.23503,7.58150,7.24133,6.78319
4,Q6SPF0;SAMD1_HUMAN,3.89547,3.19811,NaN,NaN,NaN,NaN,3.49594,3.35439,NaN,...,3.04808,NaN,3.34052,3.24627,3.91503,3.92689,3.42065,NaN,3.05477,1.89549


In [39]:
df2 = pd.read_csv('merged_proteins_metadata.csv')
df2.head()

,Project_Identifier,P37108;SRP14_HUMAN,Q96JP5;ZFP91_HUMAN,Q9Y4H2;IRS2_HUMAN,P36578;RL4_HUMAN,Q6SPF0;SAMD1_HUMAN,O76031;CLPX_HUMAN,Q8WUQ7;CATIN_HUMAN,A6NIH7;U119B_HUMAN,Q9BTD8;RBM42_HUMAN,...,Q5EBL4;RIPL1_HUMAN,P49715;CEBPA_HUMAN,Q5TA45;INT11_HUMAN,O14924;RGS12_HUMAN,Q7Z3B1;NEGR1_HUMAN,O60669;MOT2_HUMAN,Q13571;LAPM5_HUMAN,Q96JM2;ZN462_HUMAN,P35558;PCKGC_HUMAN,Tissue_type
0,SIDM00018,7.10955,3.41494,NaN,7.86661,3.89547,4.19666,NaN,NaN,3.19088,...,NaN,3.90064,2.63998,NaN,NaN,NaN,NaN,NaN,NaN,Haematopoietic and Lymphoid
1,SIDM00023,6.82802,4.14346,2.23781,7.62878,3.19811,4.60902,NaN,2.47059,3.69535,...,NaN,NaN,3.19608,NaN,NaN,NaN,NaN,NaN,NaN,Esophagus
2,SIDM00040,7.01426,4.19987,2.44055,8.12459,NaN,4.76881,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Stomach
3,SIDM00041,5.28591,3.35789,NaN,7.97268,NaN,4.52092,NaN,NaN,2.73088,...,NaN,NaN,2.79023,NaN,NaN,NaN,NaN,NaN,NaN,Stomach
4,SIDM00042,5.70786,NaN,NaN,6.22574,NaN,4.49579,NaN,NaN,2.87981,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Pancreas


In [40]:
df2_liver = df2[df2['Tissue_type'] == 'Liver']
df2_liver.head()
print(df2_liver.shape)

df2_kidney = df2[df2['Tissue_type'] == 'Kidney']
df2_kidney.head()
print(df2_kidney.shape)

df2_lung = df2[df2['Tissue_type'] == 'Lung']
df2_lung.head()
print(df2_lung.shape)

df2_ovary = df2[df2['Tissue_type'] == 'Ovary']
df2_ovary.head()
print(df2_ovary.shape)

df2_testis = df2[df2['Tissue_type'] == 'Testis']
df2_testis.head()
print(df2_testis.shape)

df2_stomach = df2[df2['Tissue_type'] == 'Stomach']
df2_stomach.head()
print(df2_stomach.shape)



(16, 6694)
(32, 6694)
(187, 6694)
(41, 6694)
(3, 6694)
(27, 6694)


In [41]:
# verwijder de tissue type kolom voor elke tissue gefilterde dataframe
df2_liver_wt = df2_liver.drop(columns=['Tissue_type'])
df2_kidney_wt = df2_kidney.drop(columns=['Tissue_type'])
df2_lung_wt = df2_lung.drop(columns=['Tissue_type'])
df2_ovary_wt = df2_ovary.drop(columns=['Tissue_type'])
df2_testis_wt = df2_testis.drop(columns=['Tissue_type'])
df2_stomach_wt = df2_stomach.drop(columns=['Tissue_type'])

In [42]:
#transpose de gefilterde dataframes voor elk tissue (zonder tissue type kolom)
df2_liver_t = df2_liver_wt.transpose().reset_index()
df2_kidney_t = df2_kidney_wt.transpose().reset_index()
df2_lung_t = df2_lung_wt.transpose().reset_index()
df2_ovary_t = df2_ovary_wt.transpose().reset_index()
df2_testis_t = df2_testis_wt.transpose().reset_index()
df2_stomach_t = df2_stomach_wt.transpose().reset_index()

df2_liver_t.head()



,index,18,441,442,464,465,466,467,468,509,851,910,911,922,923,925,938
0,Project_Identifier,SIDM00064,SIDM00585,SIDM00586,SIDM00614,SIDM00615,SIDM00616,SIDM00617,SIDM00618,SIDM00672,SIDM01110,SIDM01180,SIDM01181,SIDM01194,SIDM01196,SIDM01199,SIDM01237
1,P37108;SRP14_HUMAN,6.59723,6.29169,6.4839,6.73677,5.90851,6.53265,5.53662,5.29649,7.65011,6.90109,6.38797,6.78424,6.121,6.07664,6.06704,6.8178
2,Q96JP5;ZFP91_HUMAN,3.74143,2.83623,NaN,3.70491,2.93887,NaN,NaN,NaN,3.53629,2.33722,NaN,4.1269,NaN,NaN,NaN,NaN
3,Q9Y4H2;IRS2_HUMAN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.38012,2.30557,NaN,NaN,NaN,NaN,NaN,NaN
4,P36578;RL4_HUMAN,7.86298,7.00091,6.85553,7.27806,7.17661,7.24514,7.73104,6.93968,7.55651,7.05178,6.96467,7.90876,7.48495,7.61971,7.60702,7.60367


In [43]:
df2_liver_t.columns = df2_liver_t.iloc[0]
df2_liver_t = df2_liver_t.drop(index=0)
df2_liver_t = df2_liver_t.reset_index(drop=True)

df2_kidney_t.columns = df2_kidney_t.iloc[0]
df2_kidney_t = df2_kidney_t.drop(index=0)
df2_kidney_t = df2_kidney_t.reset_index(drop=True)

df2_lung_t.columns = df2_lung_t.iloc[0]
df2_lung_t = df2_lung_t.drop(index=0)
df2_lung_t = df2_lung_t.reset_index(drop=True)

df2_ovary_t.columns = df2_ovary_t.iloc[0]
df2_ovary_t = df2_ovary_t.drop(index=0)
df2_ovary_t = df2_ovary_t.reset_index(drop=True)

df2_testis_t.columns = df2_testis_t.iloc[0]
df2_testis_t = df2_testis_t.drop(index=0)
df2_testis_t = df2_testis_t.reset_index(drop=True)

df2_stomach_t.columns = df2_stomach_t.iloc[0]
df2_stomach_t = df2_stomach_t.drop(index=0)
df2_stomach_t = df2_stomach_t.reset_index(drop=True)


df2_liver_t.head()

,Project_Identifier,SIDM00064,SIDM00585,SIDM00586,SIDM00614,SIDM00615,SIDM00616,SIDM00617,SIDM00618,SIDM00672,SIDM01110,SIDM01180,SIDM01181,SIDM01194,SIDM01196,SIDM01199,SIDM01237
0,P37108;SRP14_HUMAN,6.59723,6.29169,6.4839,6.73677,5.90851,6.53265,5.53662,5.29649,7.65011,6.90109,6.38797,6.78424,6.121,6.07664,6.06704,6.8178
1,Q96JP5;ZFP91_HUMAN,3.74143,2.83623,NaN,3.70491,2.93887,NaN,NaN,NaN,3.53629,2.33722,NaN,4.1269,NaN,NaN,NaN,NaN
2,Q9Y4H2;IRS2_HUMAN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.38012,2.30557,NaN,NaN,NaN,NaN,NaN,NaN
3,P36578;RL4_HUMAN,7.86298,7.00091,6.85553,7.27806,7.17661,7.24514,7.73104,6.93968,7.55651,7.05178,6.96467,7.90876,7.48495,7.61971,7.60702,7.60367
4,Q6SPF0;SAMD1_HUMAN,3.77503,NaN,3.43893,3.03812,1.66604,3.19593,3.14764,NaN,2.81654,NaN,NaN,3.1304,2.91193,NaN,NaN,1.65908


In [2]:
# ──────────────────────────────────────────────────────────────────────────────
# CONFIGURATION  — only edit this block between runs
# ──────────────────────────────────────────────────────────────────────────────
DATASET      = df2_lung_t    # <-- change to: df4_lung_t, df4_kidney_t,
                              #               df4_ovary_t, df4_testis_t, df4_stomach_t
TISSUE_LABEL = "Lung"        # <-- match to dataset: "Liver", "Lung", "Kidney",
                             #                       "Ovary", "Testis", "Stomach"

SHAP_OUTPUT_FOLDER = os.path.join(
    r"C:\Users\kator\DATA\Ugent2025-2026\Master Dissertation\cell line selection\SHAP_plots",
    TISSUE_LABEL
)
# ──────────────────────────────────────────────────────────────────────────────

os.makedirs(SHAP_OUTPUT_FOLDER, exist_ok=True)

model = MLMarker(binary=False, penalty_factor=1)

prediction_dict           = {}
added_features_per_sample = {}
top3_dict                 = {}

for c in tqdm.tqdm(DATASET.columns[1:]):

    # ── Scale each cell line individually (drop NaNs first) ───────────────────
    subset = DATASET[["Project_Identifier", c]].copy()
    subset.dropna(inplace=True)

    scaler = MinMaxScaler()
    subset.loc[:, c] = scaler.fit_transform(subset[[c]])
    subset.loc[:, "Project_Identifier"] = subset["Project_Identifier"].apply(
        lambda x: x.split(";")[0]
    )
    data = subset.pivot_table(columns="Project_Identifier", values=c, aggfunc="sum")

    # ── Load sample + added features ──────────────────────────────────────────
    model.load_sample(data.iloc[0:1, :])
    added_features = model.load_sample(data.iloc[0:1, :], output_added_features=True)
    added_features_per_sample[c] = added_features

    # ── Top 100 predictions ───────────────────────────────────────────────────
    prediction = pd.DataFrame(model.predict_top_tissues_shap(n_preds=100)).T
    prediction.columns = prediction.iloc[0]
    prediction.drop(prediction.index[0], inplace=True)
    prediction = prediction.reindex(sorted(prediction.columns), axis=1)
    prediction_dict[c] = prediction.values

    safe = c.replace("/", "_").replace(" ", "_")

    # ── SHAP plot for TISSUE_LABEL ────────────────────────────────────────────
    model.shap_force_plot(tissue_name=TISSUE_LABEL)
    fig = plt.gcf()
    base = os.path.join(SHAP_OUTPUT_FOLDER, f"SHAP_{safe}_{TISSUE_LABEL}")
    fig.savefig(base + ".png", bbox_inches="tight", dpi=150)
    fig.savefig(base + ".pdf", bbox_inches="tight")
    plt.close("all")

    # ── SHAP plots for top 3 predictions ─────────────────────────────────────
    top3 = pd.DataFrame(model.predict_top_tissues_shap(n_preds=3)).T
    top3.columns = top3.iloc[0]
    top3.drop(top3.index[0], inplace=True)
    top3_tissues = top3.columns.tolist()

    top3_dict[c] = {
        "top3.1": top3_tissues[0] if len(top3_tissues) > 0 else None,
        "top3.2": top3_tissues[1] if len(top3_tissues) > 1 else None,
        "top3.3": top3_tissues[2] if len(top3_tissues) > 2 else None,
    }

    for rank, tissue in enumerate(top3_tissues, start=1):
        model.shap_force_plot(tissue_name=tissue)
        fig = plt.gcf()
        safe_tissue = tissue.replace(" ", "_")
        base = os.path.join(SHAP_OUTPUT_FOLDER, f"SHAP_{safe}_top3.{rank}_{safe_tissue}")
        base = os.path.join(SHAP_OUTPUT_FOLDER, f"SHAP_{safe}_top3.{rank}")
        fig.savefig(base + ".png", bbox_inches="tight", dpi=150)
        fig.savefig(base + ".pdf", bbox_inches="tight")
        plt.close("all")

    print(f"Done: {c}")

# ── Build predictions DataFrame ───────────────────────────────────────────────
prediction_df = pd.DataFrame({k: v.flatten() for k, v in prediction_dict.items()}).T
prediction_df.columns = prediction.columns
prediction_df = prediction_df.T
prediction_df[prediction_df < 0] = 0

# ── Build added features DataFrame ───────────────────────────────────────────
added_features_df = pd.DataFrame({
    "cell_line":        list(added_features_per_sample.keys()),
    "n_added_features": [len(v) for v in added_features_per_sample.values()],
}).set_index("cell_line")

# ── Build top 3 predictions DataFrame ────────────────────────────────────────
top3_df = pd.DataFrame(top3_dict).T
top3_df.index.name = "cell_line"

print("\n── Top 3 predicted tissues per cell line ──")
print(top3_df)

print("\n── Added features ──")
print(added_features_df)

# ── Save to CSV ───────────────────────────────────────────────────────────────
SAVE_FOLDER = r"C:\Users\kator\DATA\Ugent2025-2026\Master Dissertation\cell line selection\Predictions"
os.makedirs(SAVE_FOLDER, exist_ok=True)

prediction_df.to_csv(os.path.join(SAVE_FOLDER, f"predictions_{TISSUE_LABEL}.csv"))
added_features_df.to_csv(os.path.join(SAVE_FOLDER, f"added_features_{TISSUE_LABEL}.csv"))
top3_df.to_csv(os.path.join(SAVE_FOLDER, f"top3_predictions_{TISSUE_LABEL}.csv"))

print(f"\nSaved: predictions_{TISSUE_LABEL}.csv")
print(f"Saved: added_features_{TISSUE_LABEL}.csv")
print(f"Saved: top3_predictions_{TISSUE_LABEL}.csv")

NameError: name 'df2_lung_t' is not defined